# Unix-terminal. Работа с файлами


## Мотивация

Терминал отвязывает работу от конкретного компьютера. К серверу можно подключиться с ноутбука, домашнего компьютера и даже с телефона через SSH-клиент вроде Termius — например, проверить процесс или освободить место, пока едешь в метро. Но на маленьком экране особенно важно понимать, где находишься, что удаляешь, кому принадлежат файлы и какие процессы запущены. Команды из этого семинара — базовый язык почти всей дальнейшей работы с Linux.


## 1. Справка, `echo`, переменные и код завершения


### Справка по командам

Если забыты флаги или порядок аргументов, используют `имя_команды --help`, например `ls --help`. Полная справка открывается через `man имя_команды`: `/текст` ищет, `q` закрывает.

Для встроенных команд Bash используется `help`, например `help echo`. `which name` показывает путь к найденной команде.


In [ ]:
%%bash
which bash
help echo


### `echo`

`echo` выводит аргументы и добавляет перевод строки. `-n` убирает его, `-e` включает спецпоследовательности: `\n` — новая строка, `\t` — табуляция. Другие последовательности перечислены в `help echo`.

В одинарных кавычках текст остаётся буквальным. В двойных кавычках Bash раскрывает переменные.


In [ ]:
%%bash
course_name='Linux practice'

echo "course=$course_name"
echo 'course=$course_name'
echo -n 'without newline; '
echo -e 'first line\n\tsecond line'


### Переменные и код завершения

Имя переменной состоит из латинских букв, цифр и `_`, но не начинается с цифры. Дефисы, точки и пробелы в имени запрещены. Значение с пробелами заключают в двойные кавычки. Обычно shell-переменные называют в `snake_case`, переменные окружения — в `UPPER_SNAKE_CASE`.

Фигурные скобки отделяют имя переменной от соседнего текста: `"${group}_report.txt"`. Без скобок `$exit_code_result` означает переменную с именем `exit_code_result`, а не `$exit_code` и суффикс.

`${NAME:-default}` берёт значение `NAME`, если оно задано и не пусто, иначе использует `default`.

Каждая команда возвращает код: `0` — успех, другое значение — ошибка. `$?` содержит код последней команды. `first && second` выполняет вторую команду после успеха, `first || second` — после ошибки. `true` всегда успешна, `false` всегда завершается ошибкой.


In [ ]:
%%bash
student_group_name='ML 01'

false
exit_code=$?

echo "exit_code=$exit_code"
echo "without braces: $exit_code_result"
echo "with braces: ${exit_code}_result"
echo "report=${student_group_name}_report.txt"
echo "temporary directory=${TMPDIR:-/tmp}"

true && echo 'success'
false || echo 'failure'


### Вопрос

Пусть `group='ML 01'`. Что выведут `echo '$group'`, `echo "$group"` и `false || echo failed`?

<details>
<summary>Ответ</summary>

Первая команда выведет буквальный текст `$group`, вторая — `ML 01`, третья — `failed`, потому что `false` завершилась с ненулевым кодом.

</details>


## 2. stdin, stdout, stderr и перенаправления

Процесс — запущенный экземпляр программы. Терминал запускает процессы команд и связывает их стандартные потоки.

`cat file` читает файл и пишет его содержимое в stdout. Без имени файла `cat` читает stdin.

У процесса есть три стандартных потока:

- stdin (`0`) — входные данные;
- stdout (`1`) — обычный результат;
- stderr (`2`) — ошибки и диагностика.

`>` перезаписывает файл, `>>` дописывает, `<` подаёт файл в stdin. `2>` сохраняет stderr отдельно, `2>&1` направляет stderr туда же, куда уже направлен stdout. Перенаправления обрабатываются слева направо: `> file 2>&1` отправляет оба потока в файл, а `2>&1 > file` перенаправляет туда только stdout — stderr уже получил прежнее назначение stdout. `/dev/null` — специальный файл-приёмник: записанные туда данные отбрасываются.

Обратный слеш `\` в конце строки продолжает ту же команду на следующей строке.

stdout и stderr независимы и могут буферизоваться по-разному. После объединения их строк точный взаимный порядок не гарантирован.


In [ ]:
%%bash
input_file="${TMPDIR:-/tmp}/seminar2-streams-input.txt"
stdout_file="${TMPDIR:-/tmp}/seminar2-streams-stdout.txt"
stderr_file="${TMPDIR:-/tmp}/seminar2-streams-stderr.txt"
all_file="${TMPDIR:-/tmp}/seminar2-streams-all.txt"

echo 'first line' > "$input_file"
echo 'second line' >> "$input_file"
cat < "$input_file"

cat "$input_file" /missing-file \
  > "$stdout_file" \
  2> "$stderr_file" || true

cat "$input_file" /missing-file \
  > "$all_file" 2>&1 || true

echo 'stdout:'
cat "$stdout_file"
echo 'stderr:'
cat "$stderr_file"


### Вопрос

Команда одновременно печатает результат и сообщение об ошибке. Как сохранить их раздельно и зачем это может понадобиться?

<details>
<summary>Ответ</summary>

stdout направляют через `> result.txt`, stderr — через `2> errors.txt`. Результат можно обрабатывать дальше, а диагностику проверять отдельно.

</details>


## 3. Heredoc

Heredoc передаёт команде многострочный stdin. После `<<` указывается маркер окончания; `EOF` — только принятое имя. Закрывающий маркер должен стоять один на строке.

В `<<EOF` переменные раскрываются при создании текста. В `<<'EOF'` содержимое записывается буквально. Это тот же принцип, что у двойных и одинарных кавычек.

`echo -e 'one\ntwo'` подходит для короткого текста. Heredoc удобнее для большого блока: строки остаются читаемыми, не нужно повторять `echo` и экранировать кавычки.


In [ ]:
%%bash
short_file="${TMPDIR:-/tmp}/seminar2-heredoc-short.txt"
interpolated_file="${TMPDIR:-/tmp}/seminar2-heredoc-interpolated.txt"
literal_file="${TMPDIR:-/tmp}/seminar2-heredoc-literal.txt"

echo -e 'one\ntwo' > "$short_file"

cat > "$interpolated_file" <<EOF
USER=$USER
HOME=$HOME
EOF

cat > "$literal_file" <<'EOF'
USER=$USER
HOME=$HOME
EOF

cat "$short_file"
cat "$interpolated_file"
cat "$literal_file"


### Вопрос

В каком из файлов останется буквальный текст `$USER`: созданном через `<<EOF` или через `<<'EOF'`?

<details>
<summary>Ответ</summary>

Через `<<'EOF'`. Кавычки вокруг маркера запрещают раскрытие переменных внутри heredoc.

</details>


## 4. Переменные окружения

Полезные переменные: `$USER`, `$HOME`, `$SHELL`, `$PATH`, `$PWD`, `$OLDPWD`, `$LANG`. Полный `env` нельзя публиковать: в окружении могут быть токены.

Обычная shell-переменная видна текущей оболочке. `export` добавляет её в окружение, которое наследуют дочерние процессы.

`bash -c 'КОМАНДЫ'` запускает дочерний Bash и выполняет строку после `-c`. Одинарные кавычки оставляют `$VARIABLE` для раскрытия дочерней оболочкой.


In [ ]:
%%bash
echo "USER=$USER"
echo "HOME=$HOME"
echo "PWD=$PWD"

COURSE_EXECUTION_MODE='development'
bash -c 'echo "before export: ${COURSE_EXECUTION_MODE:-missing}"'
export COURSE_EXECUTION_MODE
bash -c 'echo "after export: $COURSE_EXECUTION_MODE"'


### Вопрос

Переменная `COURSE_NAME` задана без `export`. Что увидит `bash -c 'echo "${COURSE_NAME:-missing}"'` и как изменить результат?

<details>
<summary>Ответ</summary>

Дочерний Bash выведет `missing`. После `export COURSE_NAME` он получит значение переменной.

</details>


## 5. Типы объектов, пользователи и права


### Типы объектов

Первый символ в `ls -l` обозначает тип. Флаг `-d` показывает саму директорию, а не её содержимое, поэтому `ls -ld path` удобен для проверки типа:

- `-` — обычный файл;
- `d` — директория;
- `l` — символическая ссылка;
- `c` — символьное устройство;
- `b` — блочное устройство;
- `p` — именованный канал;
- `s` — сокет.


In [ ]:
%%bash
ls -ld /etc/os-release /tmp /dev/null


### Пользователи и группы

`whoami` показывает пользователя, `id` — UID, GID и группы, `groups` — список групп.

`chown OWNER file` меняет владельца, `chgrp GROUP file` — группу. Смена владельца обычно требует административных прав.


In [ ]:
%%bash
demo_file="${TMPDIR:-/tmp}/seminar2-owner.txt"
touch "$demo_file"

whoami
id
groups
chown "$USER" "$demo_file" 2> "${demo_file}.errors" || true
cat "${demo_file}.errors"


### Права

После типа идут три тройки прав: владелец, группа, остальные.

- `r = 4`: читать файл; видеть список имён директории;
- `w = 2`: изменять файл; создавать и удалять записи директории;
- `x = 1`: запускать файл; проходить по директории.

Директория не становится программой из-за права `x`: для неё этот бит разрешает поиск имени и проход по пути.

Числа складываются отдельно: `7=4+2+1`, `6=4+2`, `5=4+1`, `4=4`. Например, `640` — `rw-r-----`.

Для директории `r` без `x` позволяет увидеть имена, но не обратиться к ним. `x` без `r` разрешает доступ к заранее известному имени. Право `x` требуется на каждой директории пути.

`chmod 640 file` задаёт права числом, `chmod u=rw,go=r file` — символически, `chmod +x file` добавляет право запуска. `stat -c '%A %a %n'` показывает символьные и числовые права.


In [ ]:
%%bash
demo_dir="${TMPDIR:-/tmp}/seminar2-permissions"
mkdir -p "$demo_dir/shared"
echo 'report' > "$demo_dir/report.txt"

chmod 640 "$demo_dir/report.txt"
chmod 750 "$demo_dir/shared"
stat -c '%A %a %n' "$demo_dir/report.txt" "$demo_dir/shared"


### Вопрос

Что разрешают права `--x` у бинарного файла и у директории? Можно ли при `--x` увидеть список имён внутри директории?

<details>
<summary>Ответ</summary>

У бинарного файла `--x` разрешает попытку запуска. У директории — проход к заранее известному имени. Список имён без `r` увидеть нельзя; скрипту для запуска интерпретатор обычно должен ещё прочитать содержимое.

</details>


## 6. Каталоги и файлы


### Навигация и создание

`pwd` показывает текущий каталог. `cd` меняет его, `cd ..` поднимается выше, `cd -` возвращает предыдущий. `mkdir -p` создаёт дерево каталогов. `touch` создаёт пустой файл, а для существующего файла обновляет время изменения.

Файлы с точкой в начале скрыты. `ls -a` показывает их, `ls -l` включает подробный вид, `-h` делает размеры читаемыми.


In [ ]:
%%bash
demo_dir="${TMPDIR:-/tmp}/seminar2-files"
mkdir -p "$demo_dir/input" "$demo_dir/work" "$demo_dir/result"
cd "$demo_dir"

touch input/empty.txt input/.hidden
pwd
ls -lha input


### Копирование, перемещение и удаление

`cp` копирует файл, `mv` перемещает или переименовывает. `rm` удаляет файл без корзины, `rmdir` — пустой каталог, `rm -r` — каталог с содержимым. Флаг `-f` отключает подтверждения и игнорирует отсутствующие файлы.

`rm -rf` удаляет рекурсивно и без подтверждения, поэтому ошибка в пути особенно опасна. Перед удалением проверяют `pwd`, полный раскрытый путь и содержимое каталога. Нельзя передавать туда `/`, `$HOME`, пустую переменную, непроверенный `*` или пользовательский ввод. `--` завершает список флагов: `rm -- "$name"` не примет имя файла за опцию.


In [ ]:
%%bash
demo_dir="${TMPDIR:-/tmp}/seminar2-files"

cp "$demo_dir/input/empty.txt" "$demo_dir/work/"
mv "$demo_dir/work/empty.txt" "$demo_dir/work/renamed.txt"
touch "$demo_dir/result/remove-me.txt"
rm -- "$demo_dir/result/remove-me.txt"
ls -lha "$demo_dir/input" "$demo_dir/work" "$demo_dir/result"


### История и управление строкой

Стрелки `↑` и `↓` листают историю. `Ctrl+R` ищет назад, `!!` повторяет последнюю команду. `Ctrl+F` перемещает курсор вправо, а не ищет историю.

В стандартном режиме редактирования Bash `Ctrl+X` — начало сочетания, а не отдельная команда. Например, `Ctrl+X Ctrl+E` открывает текущую строку в настроенном `$EDITOR`.


### Вопрос

Почему перед `rm -rf` недостаточно проверить только имя последнего каталога в пути?

<details>
<summary>Ответ</summary>

Ошибка может находиться в любой части пути или в пустой переменной. Нужно проверить полный раскрытый путь и текущий каталог.

</details>


## 7. Процессы, jobs и сигналы

Процесс — запущенный экземпляр программы. У него есть PID, родительский PPID, состояние и ресурсы. `ps` показывает процессы системы, `jobs` — задачи текущей интерактивной оболочки.

У `ps` флаг `-p PID` выбирает процесс по PID, `-u USER` — процессы пользователя, `-o fields` — нужные поля вывода.

`&` запускает команду в фоне, `$!` содержит PID последнего фонового процесса, `wait` ожидает завершение.

`Ctrl+C` отправляет foreground-процессу SIGINT. `Ctrl+Z` приостанавливает его через SIGTSTP. `jobs` показывает номера jobs; `bg %2` продолжает вторую в фоне, `fg %2` возвращает её на передний план.

`kill PID` отправляет SIGTERM. SIGSTOP приостанавливает, SIGCONT продолжает. SIGKILL применяют только когда корректное завершение не работает.


In [ ]:
%%bash
sleep 10 > /tmp/seminar2-sleep.out 2> /tmp/seminar2-sleep.err &
process_id=$!

ps -o pid,ppid,stat,cmd -p "$process_id"
kill -STOP "$process_id"
ps -o pid,ppid,stat,cmd -p "$process_id"
kill -CONT "$process_id"
kill -TERM "$process_id"
wait "$process_id" 2>/dev/null || true
ps -p "$process_id" || echo 'process finished'


### Вопрос

В `jobs` показаны `[1]` и `[2]`, обе остановлены. Как продолжить job 1 в фоне, а job 2 вернуть на передний план?

<details>
<summary>Ответ</summary>

Выполнить `bg %1`, затем `fg %2`.

</details>


## 8. Пайплайны


`|` направляет stdout команды слева в stdin команды справа. stderr автоматически в пайплайн не попадает. Перед сборкой пайплайна каждую команду полезно проверить отдельно.

Обычно код пайплайна равен коду последней команды. `set -o pipefail` делает пайплайн неуспешным, если завершилась с ошибкой любая его часть.


In [ ]:
%%bash
echo 'pipeline data' | cat > /tmp/seminar2-pipeline.txt
cat /tmp/seminar2-pipeline.txt

cat /missing-file | cat
echo "without pipefail: $?"

set -o pipefail
cat /missing-file | cat || echo 'pipeline failed'


### Вопрос

Первая команда пайплайна завершилась с ошибкой, последняя — успешно. Как изменится код после `set -o pipefail`?

<details>
<summary>Ответ</summary>

Без `pipefail` берётся код последней команды. С `pipefail` весь пайплайн получит ненулевой код из-за ошибки одной из частей.

</details>


## 9. Система и ресурсы

`uname` показывает kernel и архитектуру, `/etc/os-release` — дистрибутив. `uname -a` выводит все основные сведения. `uptime` показывает время работы и load average, `free` — память, `df` — место на файловых системах. У `free -h` и `df -h` флаг `-h` делает размеры читаемыми.

`ps`, `top`, `htop` показывают процессы и нагрузку. `nvidia-smi` выводит состояние поддерживаемой NVIDIA GPU. Отсутствие `htop` или `nvidia-smi` нормально.

Фигурные скобки группируют команды, которые выполняются последовательно в текущем Bash. Перенаправление после `}` применяется ко всей группе. После `{` нужен пробел или перенос строки, перед `}` — перенос строки или `;`.


In [ ]:
%%bash
mkdir -p report
{
  uname -a
  cat /etc/os-release
  uptime
  free -h
  df -h "$HOME"
} > report/system.txt 2> report/errors.txt

ps -u "$USER" -o pid,ppid,stat,%cpu,%mem,cmd \
  > report/processes.txt

which top >/dev/null && echo 'top: available' || echo 'top: not available'
which htop >/dev/null && echo 'htop: available' || echo 'htop: not available'
which nvidia-smi >/dev/null && echo 'nvidia-smi: available' || echo 'nvidia-smi: not available'


### Вопрос

Нужно сохранить stdout пяти диагностических команд в `system.txt`, а все их ошибки — в `errors.txt`. Зачем здесь удобны фигурные скобки?

<details>
<summary>Ответ</summary>

Одно перенаправление после `}` применяется ко всей группе; пути не нужно повторять для каждой команды.

</details>


## 10. Первый Bash-скрипт

Последовательность команд можно записать в текстовый файл. Расширение `.sh` принято для понятности, но Bash определяет запуск не по расширению.

Первая строка `#!/usr/bin/env bash` — shebang. Она выбирает интерпретатор при прямом запуске. Строки после `#` — комментарии.

Скрипт создаётся через `<<'EOF'`, поэтому `$USER` и `$HOME` останутся внутри файла и раскроются только при его запуске.


In [ ]:
%%bash
script_file="${TMPDIR:-/tmp}/first_script.sh"
cat > "$script_file" <<'EOF'
#!/usr/bin/env bash
echo "Script started"
echo "USER=$USER"
echo "HOME=$HOME"
mkdir -p "$HOME/seminar2/script-result"
cd "$HOME/seminar2/script-result"
touch created-by-script.txt
echo "Created by Bash script" > created-by-script.txt
cp created-by-script.txt created-by-script-copy.txt
pwd
cat created-by-script-copy.txt
echo "Script finished"
EOF

cat "$script_file"


### Вопрос

Какими двумя способами запустить `first_script.sh`? Для какого из них обязательны shebang и право `x`?

<details>
<summary>Ответ</summary>

`bash first_script.sh` запускает файл через явно выбранный интерпретатор и не требует `x` или shebang. Для `./first_script.sh` нужны право `x` и корректный shebang.

</details>


## Дополнительно


### Globs

Globs раскрывает оболочка до запуска команды:

- `*` — любое количество символов;
- `?` — один символ;
- `[0-9]` — один символ из диапазона.

Обычный `*` не выбирает скрытые файлы. Кавычки запрещают раскрытие, поэтому переменную пути заключают в кавычки, а шаблон оставляют снаружи: `"$dir"/test*.sh`.

У `ls` флаг `-t` сортирует по времени, `-r` разворачивает порядок.


In [ ]:
%%bash
demo_dir="${TMPDIR:-/tmp}/seminar2-globs"
mkdir -p "$demo_dir"
touch "$demo_dir/test-one.sh" "$demo_dir/test-two.sh" "$demo_dir/.test-hidden.sh"

ls -lrth "$demo_dir"/test*.sh
ls -lrtha "$demo_dir"


### `head`, `tee` и `wc`

`head -n 5` оставляет первые пять строк. `tee file` сохраняет stdin в файл и одновременно передаёт его дальше; `tee -a file` дописывает. `wc -l` считает строки, `wc -w` — слова, `wc -c` — байты.


In [ ]:
%%bash
cat /etc/os-release \
  | tee /tmp/os-release-copy.txt \
  | head -n 5 \
  | wc -l


### Копирование каталогов

`cp -r` рекурсивно копирует каталог. Запись `source/.` означает всё содержимое `source`, включая скрытые имена, без дополнительного уровня `source`.


In [ ]:
%%bash
source_dir="${TMPDIR:-/tmp}/seminar2-copy-source"
destination_dir="${TMPDIR:-/tmp}/seminar2-copy-destination"
mkdir -p "$source_dir" "$destination_dir"
touch "$source_dir/visible.txt" "$source_dir/.hidden.txt"

cp -r "$source_dir"/. "$destination_dir"/
ls -la "$destination_dir"


### Параметры скрипта

`$0` содержит имя скрипта, `$1`, `$2` и далее — переданные аргументы, `$#` — их количество. `exit N` завершает скрипт с кодом `N`.

```bash
#!/usr/bin/env bash
echo "script=$0 arguments=$#"
echo "input=$1 label=$2"
exit 0
```

При запуске `./report.sh data.csv train` значениями `$1` и `$2` будут `data.csv` и `train`.


### Подстановка команды

`$(command)` запускает команду и подставляет её stdout в текущую строку.

```bash
current_user=$(whoami)
echo "user=$current_user"
```


### Проверки

`test "$left" -eq "$right"` сравнивает числа, `test "$left" = "$right"` — строки. `test -f path` проверяет обычный файл, `test -d path` — директорию. Команда возвращает `0` при успехе и ненулевой код при провале.

```bash
test "$(wc -c < first.txt)" -eq "$(wc -c < second.txt)"
```


### Фоновые процессы

`$!` содержит PID последнего запущенного в фоне процесса. `wait PID` ожидает именно этот процесс и возвращает его код завершения.

```bash
sleep 2 &
process_id=$!
wait "$process_id"
wait_status=$?
echo "exit_code=$wait_status"
```
